# ♻️ SmartBin — Garbage Classification (CNN + Transfer Learning)

A predictive deep-learning model that looks at a photo of a waste item and classifies it into one of
**6 bins**: `cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`.

This notebook trains and compares the three architectures taught in class:

1. **Baseline CNN** built from scratch (`Conv2D` / `MaxPooling2D` / `Dense`)
2. **Transfer learning** with **ResNet50** (ImageNet weights, frozen base + custom head)
3. **Fine-tuning** (unfreeze the deeper ResNet50 layers and retrain at a low learning rate)

The best model is exported as `garbage_model.keras` for the Streamlit app (`app.py`).

> **Dataset:** *Garbage Classification* (TrashNet) — `asdasdasasdas/garbage-classification` on Kaggle.
> ~2,527 images across 6 folders (one folder per class).


## 0. Setup
Run this on **Google Colab** with a GPU runtime (`Runtime → Change runtime type → GPU`).

In [ ]:
# Colab already has TensorFlow. kagglehub makes downloading the dataset a one-liner.
!pip install -q kagglehub

## 1. Imports

In [ ]:
import os, json, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense,
                                      Dropout, GlobalAveragePooling2D, Input)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
print('TensorFlow', tf.__version__)

## 2. Get the dataset

`kagglehub` downloads and caches the dataset, then we locate the folder that directly contains the
6 class sub-folders (the Kaggle archive nests them one level deep).

In [ ]:
import kagglehub

# Downloads to a local cache and returns the path
base_path = kagglehub.dataset_download("asdasdasasdas/garbage-classification")
print("Downloaded to:", base_path)

# Find the directory that actually holds the class folders (cardboard, glass, ...)
EXPECTED = {"cardboard", "glass", "metal", "paper", "plastic", "trash"}
DATA_DIR = None
for root, dirs, _ in os.walk(base_path):
    if EXPECTED.issubset({d.lower() for d in dirs}):
        DATA_DIR = root
        break

assert DATA_DIR is not None, "Could not locate the 6 class folders — check the dataset layout."
print("Using DATA_DIR:", DATA_DIR)
print("Classes found:", sorted(os.listdir(DATA_DIR)))

### Manual alternative (no Kaggle API)

If you'd rather not use `kagglehub`, download the dataset zip from the Kaggle page, unzip it, and set
`DATA_DIR` by hand to the folder that contains the 6 class sub-folders, e.g.:

```python
DATA_DIR = "/content/Garbage classification/Garbage classification"
```

## 3. Data generators

We rescale pixels to `[0, 1]` and apply light augmentation (rotation, flips, zoom) to the **training**
split only. The validation split gets rescale-only so we evaluate on clean images.
`validation_split=0.2` carves out 20% for validation deterministically (same `seed`).

In [ ]:
IMG_SIZE = (224, 224)   # ResNet50's native input size
BATCH_SIZE = 32
SEED = 42

train_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.2,
)
val_aug = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_ds = train_aug.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', seed=SEED, shuffle=True)

val_ds = val_aug.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', seed=SEED, shuffle=False)

# Ordered class names (index -> name)
class_indices = train_ds.class_indices
class_names = [name for name, idx in sorted(class_indices.items(), key=lambda kv: kv[1])]
NUM_CLASSES = len(class_names)
print("Class order:", class_names)

In [ ]:
# Peek at a few images to sanity-check labels
images, labels = next(train_ds)
plt.figure(figsize=(10, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i])
    plt.title(class_names[np.argmax(labels[i])])
    plt.axis('off')
plt.tight_layout(); plt.show()
train_ds.reset()

## 4. Handle class imbalance

The `trash` class has far fewer images than the others. Class weights tell the model to pay more
attention to under-represented classes during training.

In [ ]:
y_train = train_ds.classes
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight = {i: w for i, w in enumerate(weights)}
print({class_names[i]: round(w, 2) for i, w in class_weight.items()})

## 5. Approach A — Baseline CNN (from scratch)

The same architecture taught in the CNN notebook: stacked `Conv2D` + `MaxPooling2D` blocks, then a
dense classifier. This is our honest baseline.

In [ ]:
def build_cnn():
    m = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(*IMG_SIZE, 3)),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.4),
        Dense(NUM_CLASSES, activation='softmax'),
    ])
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return m

cnn = build_cnn()
cnn.summary()

In [ ]:
cnn_callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
]
hist_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=40,
                   class_weight=class_weight, callbacks=cnn_callbacks)

In [ ]:
def plot_history(h, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h.history['accuracy'], label='train')
    ax[0].plot(h.history['val_accuracy'], label='val')
    ax[0].set_title(f'{title} — accuracy'); ax[0].legend()
    ax[1].plot(h.history['loss'], label='train')
    ax[1].plot(h.history['val_loss'], label='val')
    ax[1].set_title(f'{title} — loss'); ax[1].legend()
    plt.show()

plot_history(hist_cnn, 'Baseline CNN')
cnn_val_acc = max(hist_cnn.history['val_accuracy'])
print(f"Best baseline CNN val accuracy: {cnn_val_acc:.3f}")

## 6. Approach B — Transfer learning with ResNet50

We load **ResNet50** pretrained on ImageNet without its top, **freeze** every layer, and train only a
small custom head. The pretrained filters already recognise edges, textures and materials, so this
learns our 6 classes from far fewer images than training from scratch.

In [ ]:
base_model = tf.keras.applications.ResNet50(
    weights='imagenet', include_top=False,
    input_tensor=Input(shape=(*IMG_SIZE, 3)))

for layer in base_model.layers:
    layer.trainable = False

head = base_model.output
head = GlobalAveragePooling2D()(head)
head = Dense(256, activation='relu')(head)
head = Dropout(0.3)(head)
head = Dense(NUM_CLASSES, activation='softmax')(head)

network = Model(inputs=base_model.input, outputs=head)
network.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("Trainable params (head only):",
      f"{int(np.sum([tf.size(w).numpy() for w in network.trainable_weights])):,}")

In [ ]:
tl_callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
]
hist_tl = network.fit(train_ds, validation_data=val_ds, epochs=30,
                      class_weight=class_weight, callbacks=tl_callbacks)
plot_history(hist_tl, 'Transfer learning (frozen ResNet50)')
tl_val_acc = max(hist_tl.history['val_accuracy'])
print(f"Best transfer-learning val accuracy: {tl_val_acc:.3f}")

## 7. Approach C — Fine-tuning

Now we **unfreeze the deeper layers** of ResNet50 and continue training with a **very low learning
rate** (1e-5). This gently adapts the high-level ImageNet filters to recycling materials specifically.
We keep the early, generic layers frozen.

In [ ]:
FINE_TUNE_AT = 140   # unfreeze layers from this index onward
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False
for layer in base_model.layers[FINE_TUNE_AT:]:
    layer.trainable = True

# Recompile with a low LR so we don't wreck the pretrained weights
network.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                loss='categorical_crossentropy', metrics=['accuracy'])

ft_callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
]
hist_ft = network.fit(train_ds, validation_data=val_ds, epochs=20,
                      class_weight=class_weight, callbacks=ft_callbacks)
plot_history(hist_ft, 'Fine-tuned ResNet50')
ft_val_acc = max(hist_ft.history['val_accuracy'])
print(f"Best fine-tuned val accuracy: {ft_val_acc:.3f}")

## 8. Evaluate & compare
Line up the three approaches, then inspect the winner with a confusion matrix and a full report.

In [ ]:
print("Validation accuracy comparison")
print("-" * 38)
print(f"A) Baseline CNN          : {cnn_val_acc:.3f}")
print(f"B) Transfer learning     : {tl_val_acc:.3f}")
print(f"C) Fine-tuned ResNet50   : {ft_val_acc:.3f}")

In [ ]:
# Detailed evaluation of the fine-tuned model (our deployment candidate)
val_ds.reset()
probs = network.predict(val_ds)
y_pred = np.argmax(probs, axis=1)
y_true = val_ds.classes

print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}\n")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion matrix — fine-tuned ResNet50')
plt.show()

## 9. Save the model for the app

We export the **full** fine-tuned model plus a small config file the Streamlit app reads
(class order, image size, and which classes count as recyclable).

In [ ]:
# Save whichever model scored best on validation. Default to the fine-tuned ResNet50.
best = network   # change to `cnn` if the baseline somehow wins

best.save('garbage_model.keras')

config = {
    "class_names": class_names,
    "img_size": list(IMG_SIZE),
    "recyclable": ["cardboard", "glass", "metal", "paper", "plastic"],  # 'trash' -> landfill
}
with open('class_names.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Saved garbage_model.keras and class_names.json")
print("Download both files and place them next to app.py.")

In [ ]:
# In Colab, trigger downloads of the two files the app needs:
try:
    from google.colab import files
    files.download('garbage_model.keras')
    files.download('class_names.json')
except Exception as e:
    print("Not on Colab — grab the files from the working directory.", e)

## 10. Quick single-image sanity check
Same preprocessing the app uses: resize → rescale by 1/255 → add batch dim → predict.

In [ ]:
def predict_path(path):
    img = tf.keras.preprocessing.image.load_img(path, target_size=IMG_SIZE)
    arr = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    arr = np.expand_dims(arr, 0)
    p = best.predict(arr, verbose=0)[0]
    i = int(np.argmax(p))
    print(f"Prediction: {class_names[i]}  ({p[i]*100:.1f}% confidence)")
    return class_names[i]

# Example: grab one validation image path and test it
val_ds.reset()
sample_path = os.path.join(DATA_DIR, val_ds.filenames[0])
print("Testing:", sample_path)
predict_path(sample_path)